# Project 08 · Research-and-Write Agent with Human Approval (Starter notebook)

**Module 08 · LangGraph Production Features** · Part 1 (Core) · the last project of Part 1

| | |
|---|---|
| ⏱️ Estimated time | 6–8 hours |
| 📋 Prerequisites | Lessons 08.01–08.07 (and Project 06's index, optional) |
| 🔑 Keys | `GROQ_API_KEY` (LIVE mode, optional) · `LANGSMITH_API_KEY` (traces of the fork, optional) |
| 💸 Cost (LIVE) | $0 on the free tier: one run ≈ 5–15k tokens of `gpt-oss-20b` incl. Groq browser search; the notebook makes ~4 runs (under $0.02 at paid prices) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |
| 📄 Spec | [project/README.md](README.md) (also in [COURSE_OUTLINE.md](../../../COURSE_OUTLINE.md#project-08--research-and-write-agent-with-human-approval)) |

**Goal:** a durable, interruptible research assistant: it asks when a request is vague, plans, researches (web + your docs) in a subgraph, drafts, and **waits for your approval** before it finalizes. It survives a crash, and you can fork it from the plan to try a different one.

This is the **starter**. Every `# TODO` marks work for you. The notebook runs top to bottom even before you finish: unfinished parts show ❌ in the checks instead of crashing. Stuck? Peek at the [reference solution](solutions/08-project-research-agent-solution.ipynb).

## Requirements (from the spec)

1. Graph: `clarify_question` (**interrupt** if ambiguous) → `plan` → **research subgraph** (docs search over Project 06's index + web search) → `draft` → **approval interrupt** (approve / edit / reject with feedback) → `finalize`, or back to `draft`.
2. **SQLite checkpointer**; a run survives a restart (kill it mid-run, resume).
3. **Time travel**: fork from the plan step with a different plan.
4. **Streaming** progress (node updates + tokens); **per-node retry and timeout** on the web search.

**Acceptance criteria:** survives a restart · approval is required before finalization · the fork produces a different, traceable branch (in the checkpoints, and in LangSmith with tracing on).

**Modes:** with `GROQ_API_KEY` the notebook is **LIVE** (`gpt-oss-20b` clarifies, plans and writes; Groq's built-in browser search is the web). Without it, **OFFLINE DEMO**: scripted models (`mastery.graphs.research.offline_models()`) and an offline web stand-in that returns nothing but can fail or hang **on purpose**, so retries, timeouts and the fallback are testable. The human is a parameter cell (`DECISIONS`), so Restart & Run All always completes.

| TODO | Where | What |
|---|---|---|
| 1 | Step 2 | `clarify_question`: `interrupt()` with a clarifying question when the request is ambiguous |
| 2 | Step 2 | Web search node: `RetryPolicy` + `TimeoutPolicy` + an `error_handler` that degrades to docs-only |
| 3 | Step 2 | Approval: an `approval_node` that routes approve/edit → `finalize`, reject → `draft` (max 3) |
| 4 | Step 5 | `fork_from_plan`: find the checkpoint after `plan`, `update_state(..., as_node="plan")`, run the branch with metadata |

---
## Step 0 · Set up

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/08-langgraph-production-features/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 08 can reuse Module 06): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
# Imports and settings used by the whole notebook.
import asyncio
import json
import time

from langgraph.errors import NodeError, NodeTimeoutError
from langgraph.func import task
from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime
from langgraph.types import Command, RetryPolicy, TimeoutPolicy, interrupt
from langchain_core.messages import HumanMessage, SystemMessage

from mastery.config import is_set, load_env
from mastery.graphs import timed
from mastery.graphs.hitl import adrive, approval_node, arun_until_pause, scripted_reviewer
from mastery.graphs.persistence import acheckpoint_rows, async_sqlite_checkpointer, file_size
from mastery.graphs.research import (CLARIFY_PROMPT, Clarity, Plan, ResearchContext, ResearchState, ResearchSubState,
                                     build_research_graph, corpus_docs_search, draft_prompt, offline_models,
                                     offline_web_search, planner_prompt, retriever_docs_search, run_and_kill,
                                     web_tool_search)
from mastery.memory import make_store

load_env()
LIVE = is_set("GROQ_API_KEY")
USE_PROJECT06_INDEX = True       # use Project 06's Chroma index when it exists (else BM25 over the same corpus)
RUN_CRASH_DEMO = True            # Step 4b: kill a child process mid-run (~5 s, offline)
RUN_STRETCH = True               # Step 8: preferences store (offline, free)
QUESTION = "How do I keep the battery healthy in winter?"
VAGUE = "Battery?"
OUT = REPO_ROOT / "data" / "outputs" / "m08"
OUT.mkdir(parents=True, exist_ok=True)
DB = OUT / "project08.sqlite"
FRESH_DB = True                  # False = keep the database: resume a run after a kernel restart (Step 4b box)
if FRESH_DB:                     # a fresh database every Restart & Run All
    for suffix in ("", "-wal", "-shm"):
        DB.with_name(DB.name + suffix).unlink(missing_ok=True)

if LIVE:
    from mastery.models import get_model
    from mastery.web_search import make_web_search_tool

    llm = get_model("fast", temperature=0, reasoning_effort="low")
    MODELS = {"clarifier": llm, "planner": llm, "writer": llm}
    WEB = web_tool_search(make_web_search_tool("groq", model=llm))     # 💸 Groq browser search
else:
    MODELS = offline_models()
    WEB = offline_web_search()                                          # no results; can fail/hang on purpose

DOCS, DOCS_LABEL = corpus_docs_search(), "BM25 over the Module 06 corpus"
existing = sorted((REPO_ROOT / "data" / "vectorstores").glob("project06-*")) if USE_PROJECT06_INDEX else []
if existing:
    from mastery.rag.index import default_embeddings, embedding_label, open_chroma

    emb = default_embeddings()
    collection = "project06-" + "".join(ch if ch.isalnum() else "-" for ch in embedding_label(emb).split("/")[-1].lower())[:40]
    if (REPO_ROOT / "data" / "vectorstores" / collection).is_dir():
        index = open_chroma(collection, emb)
        DOCS = retriever_docs_search(lambda q: index.similarity_search(q, k=4))
        DOCS_LABEL = f"Project 06 Chroma index ({collection})"


CLARIFICATION = "the Nimbus Pro battery in winter (below -10 °C)"   # what "you" answer to a clarifying question


def human(payload: dict):
    """A scripted human for cells that just need the run to finish: clarify → CLARIFICATION, approval → approve."""
    return CLARIFICATION if payload.get("kind") == "clarify" else "approve"


def check(label: str, ok: bool, detail: str = "") -> bool:
    print("✅" if ok else "❌", label, f"({detail})" if detail and not ok else "")
    return bool(ok)


print("Mode :", "LIVE (gpt-oss-20b + Groq browser search)" if LIVE else "OFFLINE DEMO (scripted models, no key, free)")
print("Docs :", DOCS_LABEL)

**You should now see** `Mode : OFFLINE DEMO (...)` (or LIVE with a key) and `Docs : BM25 over the Module 06 corpus` (or `Project 06 Chroma index (...)` if you built it in Project 06 and `USE_PROJECT06_INDEX = True`).

## Step 1 · Provided nodes

`plan`, `search_docs`, `draft`, `finalize` and the web search node are given (they're the same as in `mastery/graphs/research.py`). The plan goes into state as a **dict** (`model_dump()`), so checkpoints read back anywhere without an allow-list (Lesson 08.01).

In [ ]:
# Provided nodes (same as mastery/graphs/research.py). Each @timed node adds {"node", "seconds"} to run_log.
clarify_model = MODELS["clarifier"].with_structured_output(Clarity)
plan_model = MODELS["planner"].with_structured_output(Plan)
writer = MODELS["writer"]


@timed("plan")
def plan(state: ResearchState) -> dict:
    return {"plan": plan_model.invoke(planner_prompt(state["question"])).model_dump()}   # a dict in state (D-030)


@timed("search_docs")
def search_docs(state: ResearchSubState) -> dict:
    return {"findings": [f for q in state["plan"]["queries"] for f in DOCS(q)]}


@timed("draft")
def draft(state: ResearchState, runtime: Runtime[ResearchContext]) -> dict:
    prefs = ""
    if runtime.store is not None:                                     # Step 8 (stretch): long-term preferences
        user = getattr(runtime.context, "user_id", None) or "me"
        prefs = "; ".join(str(i.value.get("text", "")) for i in runtime.store.search(("preferences", user)))
    reply = writer.invoke(draft_prompt(state["question"], state["plan"], state.get("findings", []),
                                       state.get("feedback", ""), prefs))
    return {"draft": reply.text}


@timed("finalize")
def finalize(state: ResearchState) -> dict:
    sources = sorted({f["source"] for f in state.get("findings", []) if f.get("source")})
    return {"final": state["draft"].rstrip() + "\n\n---\nSources: " + (", ".join(sources) or "none")}


def make_search_web(web):
    @timed("search_web")
    async def search_web(state: ResearchSubState) -> dict:
        found = []
        for query in state["plan"]["queries"]:
            found += await asyncio.to_thread(web, query)                # blocking client in a thread; async = timeout-able
        return {"findings": found}
    return search_web

## Step 2 · Your production features
**TODO 1–3** are in the second cell below.

> 💡 **Hint:** TODO 1: move the model call into a `@task` (from `langgraph.func`) and call it from the node. A task's result is saved in the checkpoint, so when the node re-runs on resume it gets the **same** verdict back instead of asking the model again.

<details>
<summary>Still stuck on TODO 1? Show one way to write it</summary>

```python
@task
def assess_clarity(question: str) -> dict:
    return clarify_model.invoke([SystemMessage(CLARIFY_PROMPT), HumanMessage(question)]).model_dump()


@timed("clarify_question")
def clarify_question(state: ResearchState) -> dict:
    verdict = assess_clarity(state["question"]).result()
    if not verdict["ambiguous"]:
        return {}
    answer = interrupt({"kind": "clarify", "question": verdict["question"], "request": state["question"]})
    return {"clarification": str(answer), "question": f"{state['question']} ({answer})"}
```

</details>

In [ ]:
@timed("clarify_question")
def clarify_question(state: ResearchState) -> dict:
    verdict = clarify_model.invoke([SystemMessage(CLARIFY_PROMPT), HumanMessage(state["question"])])
    # TODO 1: when the request is ambiguous, pause and ask the human the model's clarifying question
    #   (payload kind "clarify"), then fold the answer into "question" and keep it in "clarification".
    #   Make it RESUME-SAFE: this node re-runs from the top on resume, so as written the model is asked
    #   again, and a different verdict would skip the pause and drop the human's answer (Lesson 08.03,
    #   rules of interrupts; Lesson 07.05 shows how a task's result is saved and replayed).
    return {}


def web_failed(state: ResearchSubState, error: NodeError) -> dict:
    """Runs when the web search failed for good: continue with docs only, and say so."""
    reason = f"{type(error.error).__name__}: {error.error}"[:200]
    return {"findings": [{"kind": "web", "query": "*", "text": f"Web search unavailable ({reason}).", "source": None}],
            "run_log": [{"node": "search_web", "seconds": 0.0, "error": reason}]}


def web_node_options(timeout_s: float, attempts: int) -> dict:
    """Keyword arguments for add_node("search_web", ...)."""
    # TODO 2: return {"timeout": TimeoutPolicy(run_timeout=timeout_s),
    #                 "retry_policy": RetryPolicy(max_attempts=attempts, initial_interval=0.2, jitter=False,
    #                                             retry_on=(ConnectionError, TimeoutError, NodeTimeoutError)),
    #                 "error_handler": web_failed}
    return {}


def make_approval(max_rejections: int = 3):
    """The approval node and its possible destinations."""
    # TODO 3: return approval_node(approved="finalize", rejected="draft", max_rejections=max_rejections,
    #                              give_up="finalize"), ("finalize", "draft")
    def auto_approve(state: ResearchState) -> Command:       # placeholder: NO human in the loop (fails the checks)
        return Command(update={"approval": "auto"}, goto="finalize")
    return auto_approve, ("finalize",)

In [ ]:
def build_research(web=WEB, timeout_s: float = 30, attempts: int = 3):
    """The research subgraph: its own schema; docs first, then web (sequential: see Lesson 08.02's caveat)."""
    sub = StateGraph(ResearchSubState)
    sub.add_node("search_docs", search_docs)
    sub.add_node("search_web", make_search_web(web), **web_node_options(timeout_s, attempts))
    sub.add_edge(START, "search_docs")
    sub.add_edge("search_docs", "search_web")
    sub.add_edge("search_web", END)
    return sub.compile(name="research")


def build_agent(*, checkpointer=None, store=None, web=WEB, timeout_s: float = 30, attempts: int = 3):
    research_graph = build_research(web, timeout_s, attempts)

    async def research(state: ResearchState) -> dict:          # Way 2 (Lesson 08.05): return only the new items
        out = await research_graph.ainvoke({"plan": state["plan"]})
        return {"findings": out.get("findings", []), "run_log": out.get("run_log", [])}

    approval, destinations = make_approval()
    b = StateGraph(ResearchState, context_schema=ResearchContext)
    b.add_node("clarify_question", clarify_question)
    b.add_node("plan", plan)
    b.add_node("research", research)
    b.add_node("draft", draft)
    b.add_node("approval", approval, destinations=destinations)
    b.add_node("finalize", finalize)
    b.add_edge(START, "clarify_question")
    b.add_edge("clarify_question", "plan")
    b.add_edge("plan", "research")
    b.add_edge("research", "draft")
    b.add_edge("draft", "approval")
    b.add_edge("finalize", END)
    return b.compile(checkpointer=checkpointer, store=store, name="research_agent")


saver = await async_sqlite_checkpointer(DB)       # async graph (timeouts) → AsyncSqliteSaver
agent = build_agent(checkpointer=saver)
print([n for n in agent.get_graph().nodes if not n.startswith("__")])
print("subgraph:", [n for n in agent.get_graph().nodes if n.startswith("__")] and "research = search_docs → search_web")

**You should now see** the six parent nodes `['clarify_question', 'plan', 'research', 'draft', 'approval', 'finalize']` and `subgraph: research = search_docs → search_web`. `show_graph(agent)` (Module 07) draws the parent; the research subgraph is called from inside the `research` node (Way 2, Lesson 08.05), so it appears as one node.

## Step 3 · Run it: stream, pause, answer

In [ ]:
# Stream node updates (parent + subgraph) and the draft's tokens until the first pause (LIVE: real model tokens).
from IPython.display import Markdown, display


async def stream_until_pause(agent, inputs, cfg, context=None):
    view = display(Markdown("⏳ starting…"), display_id=True)
    steps, text = [], ""
    async for part in agent.astream(inputs, cfg, stream_mode=["updates", "messages"], subgraphs=True,
                                    version="v2", context=context):
        where = " > ".join(ns.split(":")[0] for ns in part["ns"])
        if part["type"] == "updates":
            steps += [f"{where + ' > ' if where else ''}{node}" for node in part["data"]]
        elif part["data"][1].get("langgraph_node") == "draft":
            text += part["data"][0].content
        view.update(Markdown("✅ " + " → ".join(steps) + (f"\n\n**Draft:** {text[-400:]}▌" if text else "")))
    state = await agent.aget_state(cfg)
    view.update(Markdown("✅ " + " → ".join(steps) + ("\n\n⏸️ **waiting for you**" if state.interrupts else "\n\n🏁 done")))
    print("steps   :", " → ".join(steps))
    print("streamed:", len(text), "draft characters")
    return state


CFG = {"configurable": {"thread_id": "main"}, "metadata": {"branch": "main"}, "tags": ["project08", "main"]}
state = await stream_until_pause(agent, {"question": VAGUE}, CFG)
for i in state.interrupts:
    print("PAUSED  :", i.value.get("kind"), "→", i.value.get("question"))

**You should now see** a live box, `steps   : assess_clarity → __interrupt__` (the clarifier task finished and was saved; the node then paused), `streamed: 0 draft characters` and `PAUSED  : clarify → Which product or topic do you mean, and for whom?` (solution): the scripted clarifier finds `Battery?` too vague. LIVE, the model asks its own question. Starter before TODO 1: no clarify pause; the run goes straight to the end (auto-approve) and the next cells have nothing to answer.

In [ ]:
# Parameters: your approval decisions, in order (a clarifying question gets CLARIFICATION from Step 0).
# Try {"action": "edit", "text": "..."} or more rejections. Max 3 rejections, then it finalizes as "gave_up".
DECISIONS = [{"action": "reject", "feedback": "shorter"}, "approve"]

In [ ]:
# Answer each pause in order. A clarify pause gets CLARIFICATION, approval pauses get DECISIONS.
answers = list(DECISIONS)
for _ in range(10):
    state = await agent.aget_state(CFG)
    if not state.interrupts:
        break
    payload = state.interrupts[0].value
    answer = CLARIFICATION if payload.get("kind") == "clarify" else (answers.pop(0) if answers else "approve")
    print(f"⏸️ {payload.get('kind')}: answering {answer!r}")
    await arun_until_pause(agent, Command(resume=answer), CFG, printer=None)

main = (await agent.aget_state(CFG)).values
print("approval:", main.get("approval"), "| rejections:", main.get("rejections", 0))
print(main.get("final", "(not finalized)")[:700])

**You should now see** (solution) `⏸️ clarify: answering '...'`, `⏸️ approval: answering {'action': 'reject', 'feedback': 'shorter'}`, `⏸️ approval: answering 'approve'`, then `approval: approved | rejections: 1` and the final answer: two bullets (the scripted writer applied "shorter"), `_Revised after review: shorter_`, and a `Sources:` line. Starter: no `⏸️` lines, `approval: auto | rejections: 0`; nobody was asked.

## Step 4 · Survive a restart

In [ ]:
# (a) A "restart" in this kernel: pause a NEW thread at approval, close the database, reopen it, resume.
RCFG = {"configurable": {"thread_id": "restart"}}
result = await arun_until_pause(agent, {"question": QUESTION}, RCFG, printer=None)
while result["interrupts"] and result["interrupts"][0].value.get("kind") == "clarify":   # LIVE models may ask first
    result = await arun_until_pause(agent, Command(resume=CLARIFICATION), RCFG, printer=None)
paused_at = (await agent.aget_state(RCFG)).next
await saver.conn.close()                                   # the "process" ends: nothing left in memory

saver = await async_sqlite_checkpointer(DB)                # a new process opens the same file
agent = build_agent(checkpointer=saver)
after = await adrive(agent, Command(resume="approve"), RCFG, human)
print("paused at:", paused_at, "→ after reopening:", "finalized" if "final" in after["values"] else after["values"].get("approval"))
print("database :", file_size(DB))

**You should now see** `paused at: ('approval',) → after reopening: finalized` (LIVE: a clarifying question, if the model asks one, is answered with `CLARIFICATION` first) and the database size (100–250 KB). Starter: `paused at: ()`; the thread never paused, so there was nothing to survive. Nothing about the paused run was in memory after `close()`; it all came from `project08.sqlite`.

In [ ]:
# (b) A real crash, as a runtime DEMO (not graded): the REFERENCE graph runs in a child process, gets
# killed while it searches, and is resumed HERE. It shows what durable execution gives you; your own
# graph's persistence is graded by the restart check in (a).
if RUN_CRASH_DEMO:
    crash_db = OUT / "project08-crash.sqlite"
    for suffix in ("", "-wal", "-shm"):
        crash_db.with_name(crash_db.name + suffix).unlink(missing_ok=True)
    print("\n".join(run_and_kill(crash_db, "crash", QUESTION, durability="sync", slow=3)))
    crash_saver = await async_sqlite_checkpointer(crash_db)
    reference = build_research_graph(offline_models(), web_search=offline_web_search(), docs_search=corpus_docs_search(),
                                     checkpointer=crash_saver)
    ccfg = {"configurable": {"thread_id": "crash"}}
    next_after_crash = (await reference.aget_state(ccfg)).next
    print("next after the crash:", next_after_crash, "← finished nodes were saved; only research is left")
    crashed = await adrive(reference, None, ccfg, scripted_reviewer(["approve"]))
    print("resumed and finished:", crashed["done"], "| nodes:", [r["node"] for r in crashed["values"]["run_log"]])
    await crash_saver.conn.close()
else:
    crashed = None
    print("⏭️ Skipped (RUN_CRASH_DEMO = False)")

**You should now see** the child's `PID`, `STEP clarify_question`, `STEP plan`, the slow search line, `💥 killed`, then `next after the crash: ('research',)` and `resumed and finished: True | nodes: ['clarify_question', 'plan', 'search_docs', 'search_web', 'draft', 'finalize']`. The child used `durability="sync"` (Lesson 08.01 measured why).

> **Kill the kernel yourself once:** set `FRESH_DB = False` in Step 0 (otherwise Step 0 deletes the database you want to resume), run Step 3, and kill the kernel while it streams (**Kernel → Restart** mid-run). Then run Step 0, the cells that define and build `agent`, and `print((await agent.aget_state(CFG)).next)`: the finished nodes are still there. Finish with `await arun_until_pause(agent, None, CFG)`. Set `FRESH_DB = True` again afterwards.

## Step 5 · Time travel: fork from the plan

**TODO 4** is `fork_from_plan` in the next cell.

In [ ]:
NEW_PLAN = {"queries": ["mount the station outdoors safely"], "outline": ["Where to mount", "How to mount"]}


async def fork_from_plan(agent, thread_cfg: dict, new_plan: dict, label: str = "fork-1") -> dict | None:
    """Fork the thread right after `plan` with new_plan; run the branch to the end (auto-approve). Return its values."""
    # TODO 4:
    #   history = [s async for s in agent.aget_state_history(thread_cfg)]
    #   after_plan = next(s for s in history if s.next == ("research",))
    #   fork_cfg = await agent.aupdate_state(after_plan.config, {"plan": new_plan}, as_node="plan")
    #   meta = {"metadata": {"branch": label, "forked_from": after_plan.config["configurable"]["checkpoint_id"]},
    #           "tags": ["project08", "fork"]}
    #   await arun_until_pause(agent, None, {**fork_cfg, **meta}, printer=None)
    #   out = await adrive(agent, Command(resume="approve"), {**thread_cfg, **meta}, scripted_reviewer([]))
    #   return out["values"]
    return None


THREAD = {"configurable": {"thread_id": "main"}}
fork = await fork_from_plan(agent, THREAD, NEW_PLAN)
if fork:
    print("main  :", main["plan"]["queries"][0], "|", sorted({f["source"] for f in main["findings"] if f.get("source")}))
    print("fork-1:", fork["plan"]["queries"][0], "|", sorted({f["source"] for f in fork["findings"] if f.get("source")}))
    labels = sorted({s.metadata.get("branch", "?") async for s in agent.aget_state_history(THREAD)})
    print("branch labels in the thread:", labels)
else:
    print("(no fork yet: TODO 4)")

**You should now see** (solution) the main branch's first query (the battery question, with its clarification) and battery sources, the fork's `mount the station outdoors safely` with mounting sources, and `branch labels in the thread: ['?', 'fork-1', 'main']` (`?` = the checkpoints written by the resume cells, which had no metadata). Starter: `(no fork yet: TODO 4)`.

**In LangSmith** (tracing on): filter the project by tag `fork` or `metadata.branch = fork-1`. The fork's trace starts at `research`; the main trace starts at `clarify_question`.

## Step 6 · Retry, timeout, fallback

In [ ]:
# Retry, timeout and fallback on the web search, with offline stand-ins (free, deterministic, no model calls):
# we run only the research subgraph, with a fixed plan.
FIXED_PLAN = {"queries": [QUESTION, QUESTION + " troubleshooting"], "outline": ["What to know", "What to do"]}
flaky = offline_web_search(failures=1)                  # fails once, then works → retried, no fallback
down = offline_web_search(failures=99)                  # always fails → retries exhausted → error_handler
slow = offline_web_search(delay=1.0)                    # hangs → timeout → retried → error_handler
policy_runs = {}
for name, web, timeout_s in (("flaky", flaky, 30), ("down", down, 30), ("slow", slow, 0.3)):
    research_graph = build_research(web, timeout_s=timeout_s, attempts=2)
    start = time.perf_counter()
    try:
        out = await research_graph.ainvoke({"plan": FIXED_PLAN})
        notes = [f["text"] for f in out.get("findings", []) if f["kind"] == "web"]
        policy_runs[name] = {"ok": True, "web_calls": web.calls["calls"], "web_notes": notes}
    except Exception as exc:                            # starter without TODO 2: the run fails
        policy_runs[name] = {"ok": False, "web_calls": web.calls["calls"], "error": type(exc).__name__}
    print(f"{name:<5} {time.perf_counter() - start:4.1f}s", policy_runs[name])

**You should now see** (solution):

- `flaky` ok with `web_calls: 3` (1 failed call, then 2 queries) and no web notes (the offline web returns nothing once it works); the retry made the failure invisible;
- `down` ok after 2 attempts, with `Web search unavailable (ConnectionError: ...)`;
- `slow` ok in under a second, with `Web search unavailable (NodeTimeoutError: ...)`: each attempt was cut at 0.3 s.

Starter before TODO 2: `flaky` and `down` fail with `ConnectionError`; `slow` "works" but takes 2+ s (no timeout).

## Step 7 · Acceptance checks

In [ ]:
# Acceptance checks. ✅ = done; ❌ shows which TODO to look at.
ok = []
rows = await acheckpoint_rows(agent, {"configurable": {"thread_id": "main"}})
ok.append(check("survives a restart: a thread paused before the reopen finalized after it",
                paused_at == ("approval",) and "final" in after["values"], "TODO 3"))
if crashed is None:
    print("⏭️ crash demo skipped (RUN_CRASH_DEMO = False): not graded either way")
else:   # The demo uses the reference graph, so it's reported, not counted.
    print("ℹ️ crash demo (reference graph, not graded):",
          "resumed from SQLite and finished" if crashed["done"] and next_after_crash == ("research",) else "did NOT resume as expected")
ok.append(check("a vague request triggers a clarifying question", main.get("clarification") == CLARIFICATION, "TODO 1"))
if not LIVE:   # the scripted clarifier records its calls: one call = the verdict was replayed on resume
    asked = sum(1 for c in MODELS["clarifier"].calls if c and c[-1].text == VAGUE)
    ok.append(check(f"clarify is resume-safe: the model judged the vague request once, not again on resume ({asked}×)",
                    asked == 1 and main.get("clarification") == CLARIFICATION, "TODO 1"))
approval_pauses = [r for r in rows if r["next"] == ("approval",) and r["interrupts"]]
ok.append(check("approval is required: the run paused for a human before finalize",
                bool(approval_pauses) and main.get("approval") in {"approved", "edited", "gave_up"}, "TODO 3"))
ok.append(check("reject sends it back to draft with the feedback",
                main.get("rejections", 0) == sum(1 for d in DECISIONS if isinstance(d, dict) and d.get("action") == "reject"),
                "TODO 3"))
fork_ok = bool(fork) and fork["plan"] != main["plan"] and fork.get("final") != main.get("final")
ok.append(check("the fork from the plan produced a different branch", fork_ok, "TODO 4"))
labels = {s.metadata.get("branch") async for s in agent.aget_state_history({"configurable": {"thread_id": "main"}})}
ok.append(check("the branch is traceable (metadata branch/forked_from on its checkpoints)", {"main", "fork-1"} <= labels, "TODO 4"))
ok.append(check("web search retries a transient failure", policy_runs["flaky"]["ok"] and policy_runs["flaky"]["web_calls"] >= 2
                and not policy_runs["flaky"]["web_notes"], "TODO 2"))
ok.append(check("a dead or hanging web search degrades to docs-only",
                all(policy_runs[n]["ok"] and any("unavailable" in t for t in policy_runs[n]["web_notes"]) for n in ("down", "slow")),
                "TODO 2"))
print(f"\n{sum(ok)}/{len(ok)} acceptance checks passed")
json.dump({"mode": "LIVE" if LIVE else "OFFLINE", "checks": f"{sum(ok)}/{len(ok)}", "checkpoints_main": len(rows),
           "policy_runs": policy_runs}, open(OUT / "project08-report.json", "w", encoding="utf-8"), indent=1, default=str)

**You should now see** `9/9 acceptance checks passed` (solution, offline; LIVE runs 8: the resume-safety check needs the scripted clarifier) and an `ℹ️ crash demo` line. Starter: `0/9`, with ❌ next to the TODO to fix. The report is saved to `data/outputs/m08/project08-report.json`.

## Step 8 · Stretch goals

In [ ]:
# Stretch: long-term writing preferences in a store, read by `draft` (Lesson 08.07).
if RUN_STRETCH:
    prefs = make_store("memory")
    prefs.put(("preferences", "ana"), "style", {"text": "one paragraph, no bullet lists"})
    styled = build_agent(checkpointer=saver, store=prefs)
    outs = {}
    for user in ("ana", "ben"):
        out = await adrive(styled, {"question": QUESTION}, {"configurable": {"thread_id": f"prefs-{user}"}},
                           human, context=ResearchContext(user_id=user))
        outs[user] = out["values"]["final"]
        body = out["values"]["draft"].split("\n\n", 1)[-1]           # skip the "# title" line
        print(f"{user}: {body[:120].replace(chr(10), ' ')}...")
else:
    print("⏭️ Skipped (RUN_STRETCH = False)")

**You should now see** Ana's answer as one paragraph and Ben's as bullets: same graph, same question, different stored preferences.

**More stretch ideas:**

- **Agent Chat UI:** [Agent Chat UI](https://docs.langchain.com/oss/python/langchain/ui) talks to chat graphs (a `messages` key). This graph uses `question`, so write a thin wrapper graph with `MessagesState` that takes the last human message as the question, calls this agent as a subgraph (Way 2), and returns the final answer as an AI message; serve it from `apps/m08_research_agent/` under a second graph id and connect the UI to `http://127.0.0.1:2024`. Until then, answer the interrupts in **Studio**.
- Swap `InMemorySaver`-style demo stores for `make_store("sqlite", ...)` so preferences survive restarts too.
- Give the clarifier a budget: at most one clarification per thread.

In [ ]:
await saver.conn.close()
print("closed", DB.relative_to(REPO_ROOT), "|", file_size(DB))

## Findings (write yours)

| | Main branch | Fork |
|---|---|---|
| plan (first query) | | |
| sources used | | |
| rejections before approval | | |
| checkpoints / database size | | |

- What did your clarifying question add? Would you ask it, as a human expert?
- Restart vs crash: which steps were repeated after the crash, and why is that safe here?
- Which durability mode would you ship for this agent, and why? (Lesson 08.01's measurements.)
- Is the fork a fair comparison? What else would you keep fixed?
- Log decisions in [`docs/decision-log.md`](../../../docs/decision-log.md).

## ✅ Before you commit

- [ ] Restart & Run All passes (LIVE and without a key).
- [ ] All TODOs done; 9/9 acceptance checks (8/8 LIVE); findings written.
- [ ] I ran the app under `langgraph dev`, answered the interrupts in Studio, and forked a thread there once.
- [ ] Outputs cleared (`jupyter nbconvert --clear-output --inplace <notebook>`); no keys in outputs; `data/outputs/` not committed.

---
## ➡️ What's next

🎉 **This was the last project of Part 1 (Core).** You can now build agents with LangChain and production-grade graphs with LangGraph: persistence, interrupts, time travel, subgraphs, streaming and stores. See the [Part 1 wrap-up](../../../modules/08-langgraph-production-features/README.md#part-1-complete) in the Module 08 README.

**Next:** Part 2 (Pro) starts with **[Module 09 · Multi-Agent Systems](../../../modules/09-multi-agent-systems/README.md)**: subagents, handoffs, skills, routers and custom workflows, compared by model calls, tokens and latency, built on the subgraphs, state and interrupts you just used. Start with its README's **Before you start this module** block (pull, install, verify).